<a href="https://colab.research.google.com/github/maryamyazdi/Sentiment-Analysis/blob/main/new.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!pip install hazm
import hazm as hz
import re
import pickle
import os
import torch
import numpy as np
from torch.utils.data import DataLoader, Dataset
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

In [8]:
df = pd.read_csv('/content/drive/My Drive/train.csv', sep='\t', index_col=False)
train_data = (df.head(n=300)).drop(axis=1, columns='Unnamed: 0')

def fixup(x):
    x = x.replace('\u200c', '').replace('\xa0','').replace('\r\n',' ').replace('|',' ')
    return x

normalizer = hz.Normalizer()
def my_tokenizer(text):
  text = re.sub(r"[\{\}\؛\*\=\-\+\/\n\(\)]"," ",str(text))
  text = re.sub("[ ]+"," ",text)
  text = re.sub("\!+","!",text)
  text = re.sub("[؟]+","؟",text)
  text = re.sub("\?+","?",text)
  text = re.sub("[.]+",".",text)
  #text = re.sub("[ر]+","ر",text)
  text = fixup(normalizer.normalize(text))
  sentences = hz.sent_tokenize(text)
  for every_sentence in sentences:
    words=hz.word_tokenize(every_sentence)
  return words

tokens = []
num_words = []

for comment in train_data['comment']:
  tokens.append(my_tokenizer(comment))
  num_words.append(len(tokens[-1]))

In [10]:
  # print statistics
print('Min length =', min(num_words))
print('Max length =', max(num_words))
print('Mean = {:.2f}'.format(np.mean(num_words)))
print('Std  = {:.2f}'.format(np.std(num_words)))
print('mean + 2 * sigma = {:.2f}'.format(np.mean(num_words) + 2.0 * np.std(num_words)))

Min length = 1
Max length = 126
Mean = 12.81
Std  = 10.90
mean + 2 * sigma = 34.61


In [9]:
class Vocabulary(object):
    
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.word2index = {}
        self.word2count = {}
        self.index2word = {}
        self.count = 0
    
    def add_word(self, word):
        if not word in self.word2index:
            self.word2index[word] = self.count
            self.word2count[word] = 1
            self.index2word[self.count] = word
            self.count += 1
        else:
            self.word2count[word] += 1
    
    def add_sentence(self, sentence):
        for word in self.tokenizer(sentence):
            self.add_word(word)

In [11]:
PAD = '<pad>'
UNK = '<unk>'

class TextClassificationDataset(Dataset):

    def __init__(self, tokenizer, vocab_path, max_len , min_count):
  
        self.vocab_path = vocab_path
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.min_count = min_count
        
        self.cache = {}
        self.vocab = None
        
        self.classes = []
        self.class_to_index = {}
            
        # build vocabulary from training and validation texts
        self.build_vocab()
        
    def __getitem__(self, index):
        # read the tokenized text file and its label (neg=0, pos=1)

        # if fname in self.cache:
        #   #return self.cache[fname], class_idx
        #   print("done")
        #   return

        # read text file 
        #text = open(fname).read()
        text = train_data['comment'][0]

        # tokenize the text file
        tokens = self.tokenizer(text)
        
        # padding and trimming
        if len(tokens) < self.max_len:
            num_pads = self.max_len - len(tokens)
            tokens = [PAD] * num_pads + tokens
        elif len(tokens) > self.max_len:
            tokens = tokens[:self.max_len]
            
        # numericalizing
        ids = torch.LongTensor(self.max_len)
        for i, word in enumerate(tokens):
            if word not in self.vocab.word2index:
                ids[i] = self.vocab.word2index[UNK]  # unknown words
            elif word != PAD and self.vocab.word2count[word] < self.min_count:
                ids[i] = self.vocab.word2index[UNK]  # rare words
            else:
                ids[i] = self.vocab.word2index[word]
                
        # save in cache for future use
        self.cache[fname] = ids
        
        return ids
    
    def build_vocab(self):
        if not os.path.exists(self.vocab_path):
            vocab = Vocabulary(self.tokenizer)
            with open('/content/drive/My Drive/sss.csv', encoding='utf8') as f:
              for line in f:
                vocab.add_sentence((line.split(sep='\t'))[1])

            # sort words by their frequencies
            words = [(0, PAD), (0, UNK)]
            words += sorted([(c, w) for w, c in vocab.word2count.items()], reverse=True)

            self.vocab = Vocabulary(self.tokenizer)
            for i, (count, word) in enumerate(words):
                self.vocab.word2index[word] = i
                self.vocab.word2count[word] = count
                self.vocab.index2word[i] = word
                self.vocab.count += 1

            pickle.dump(self.vocab, open(self.vocab_path, 'wb'))
        else:
            self.vocab = pickle.load(open(self.vocab_path, 'rb'))

In [12]:
min_count = 10
vocab_path = 'vocab.pkl'
max_len = np.mean(num_words) + 2.0 * np.std(num_words)
train_ds = TextClassificationDataset( my_tokenizer, vocab_path, max_len, min_count)

vocab = train_ds.vocab
freqs = [(count, word) for (word, count) in vocab.word2count.items() if count >= min_count]
vocab_size = len(freqs) + 2  # for PAD and UNK tokens
print(f'Vocab size = {vocab_size}')

print('\nMost common words:')
for c, w in sorted(freqs, reverse=True)[:10]:
    print(f'{w}: {c}')

Vocab size = 64

Most common words:
بود: 166
و: 158
،: 65
هم: 61
به: 59
که: 58
غذا: 53
خیلی: 53
.: 53
با: 51
